In [2]:
"""
This program search through the entire SAS platform to look for any SAS file which contains the search string.
Author : Margaret Woon
Version : 0.001
Date : 2024/06/15
"""
import os
import concurrent.futures
from datetime import datetime

def search_string_in_file(file_path, search_string):
    matches = []
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
            for line_number, line in enumerate(file, start=1):
                if search_string.lower() in line.lower():
                    matches.append((line_number, line.strip()))
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
    if matches:
        return file_path, matches
    return None

def scan_directory(directories, search_string):
    sas_files = []
    for directory in directories:
        for root, dirs, files in os.walk(directory):
            for file in files:
                if file.endswith('.sas'):
                    sas_files.append(os.path.join(root, file))

    matched_files = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        future_to_file = {executor.submit(search_string_in_file, file, search_string): file for file in sas_files}
        for future in concurrent.futures.as_completed(future_to_file):
            file = future_to_file[future]
            try:
                result = future.result()
                if result:
                    matched_files.append(result)
            except Exception as e:
                print(f"Error processing file {file}: {e}")
   
    return matched_files

datetime_now = datetime.now().strftime("%d/%m/%Y, %H:%M:%S")
                        
print(f"Job Started at {datetime_now}\n")

#
# String to search
#
string_to_search = 'PRODUCT_DEPOSIT_OFFSET_IND'

print(f"Searching for {string_to_search}\n")
      
# Places to search :
directories_to_scan = ['//sasnasprd001/PRD_Warehouse/05 Deployed Jobs/',
 '//sasnasprd001/PRD_Warehouse_legacy/05 Deployed Jobs/',
'//sasnasprd001/PRD_Warehouse/10 Stored Process/',
'F:/Lev1/DIReporting/SASEnvironment/SASCode/Jobs/',
 '//sasnasprd001/PRD_Environment/Jobs/Operational_Reporting/',
 '//dc1APfls001/groups/Finance/14. Data Warehouse Source Files/Projects/Business Warehouse/Analysis - Other/Apra/SAS Code/SAS EFS Report programs/*.sas',
 '//virwsapp258/EDAO/05 Deployed Jobs/',
 '//sasnasprd001/prd_Environment/User_Written/',
 '//sascompprd002/PRD_Environment/Jobs/',
 '//sascompprd002/PRD_Environment/User_Written/']

# Initialize the output file
outputfile = f'platform_search_result_{string_to_search}.txt'

# Make sure starts with a clean file
file_to_delete = open(outputfile, "w")
file_to_delete.close()

if __name__ == "__main__":   

    matching_files = scan_directory(directories_to_scan, string_to_search)

    if matching_files:
        with open(outputfile,'w', encoding='utf-8') as output_file:
            for file_path, lines in matching_files:
                output_file.write(f"{file_path}:\n")
                output_file.write(f"{'='*len(file_path)}\n\n")
                for line_number, line in lines:
                    output_file.write(f"Line {line_number}: {line}\n")
                output_file.write(f"\n\n")
        # Print a success message
        print(f"Outcomes : {outputfile}\n")
    else:
        print(f"No files found with the string {string_to_search}\n")

        
datetime_now = datetime.now().strftime("%d/%m/%Y, %H:%M:%S")
                        
print(f"Job Completed at {datetime_now}\n")

Job Started at 15/09/2025, 09:24:07

Searching for PRODUCT_DEPOSIT_OFFSET_IND

Outcomes : platform_search_result_PRODUCT_DEPOSIT_OFFSET_IND.txt

Job Completed at 15/09/2025, 09:26:04

